In [2]:
# Install required packages in Colab
!pip install chromadb python-dotenv groq langchain sentence-transformers langchain-community

In [ ]:

import os
import chromadb
from chromadb.utils import embedding_functions
from groq import Groq
from langchain_community.embeddings import HuggingFaceEmbeddings

# ✅ Step 1: Set your Groq API key directly in Colab
# Replace with your actual key
os.environ["GROQ_API_KEY"] = ""

# ✅ Step 2: Initialize Groq client using the environment variable
groq_key = os.environ["GROQ_API_KEY"]
client = Groq(api_key=groq_key)

# ✅ Step 3: Use Groq embeddings (Note: This specific embedding function for ChromaDB is being removed for consistency)
# groq_ef = embedding_functions.OpenAIEmbeddingFunction(
#     api_key=groq_key, model_name="text-embedding-3-small"
# )

# Initialize Chroma client with persistence
chroma_client = chromadb.PersistentClient(path="chroma_persistent_storage")
collection_name = "document_qa_collection"
# Removed embedding_function from collection creation, as we will provide embeddings manually
collection = chroma_client.get_or_create_collection(name=collection_name)

# Function to load documents
def load_documents_from_directory(directory_path):
    print("==== Loading documents from directory ====")
    documents = []
    for filename in os.listdir(directory_path):
        if filename.endswith(".txt"):
            with open(os.path.join(directory_path, filename), "r", encoding="utf-8") as file:
                documents.append({"id": filename, "text": file.read()})
    return documents

# Function to split text into chunks
def split_text(text, chunk_size=1000, chunk_overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return chunks


# Load and split documents
directory_path = "./news_articles"
documents = load_documents_from_directory(directory_path)
print(f"Loaded {len(documents)} documents")

chunked_documents = []
for doc in documents:
    chunks = split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_documents.append({"id": f"{doc['id']}_chunk{i+1}", "text": chunk})

# Initialize HuggingFaceEmbeddings once for all embedding needs
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Function to generate embeddings using Groq (kept as is, but not used for ChromaDB operations)
def get_groq_embedding(text):
    response = client.embeddings.create(input=text, model="text-embedding-3-small")
    embedding = response.data[0].embedding
    return embedding

# Generate embeddings for chunks using HuggingFaceEmbeddings
for doc in chunked_documents:
    doc["embedding"] = embeddings.embed_query(doc["text"])
# Upsert into Chroma with generated embeddings
for doc in chunked_documents:
    collection.upsert(
        ids=[doc["id"]], documents=[doc["text"]], embeddings=[doc["embedding"]]
    )

# Query function - now generates embeddings for the question using HuggingFaceEmbeddings
def query_documents(question, n_results=2):
    # Embed the query text using the same HuggingFaceEmbeddings instance
    query_embedding = embeddings.embed_query(question)
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results)
    relevant_chunks = [doc for sublist in results["documents"] for doc in sublist]
    return relevant_chunks

# Generate response using Groq
def generate_response(question, relevant_chunks):
    context = "\n\n".join(relevant_chunks)
    prompt = (
        "You are an assistant for question-answering tasks. Use the following pieces of "
        "retrieved context to answer the question. If you don't know the answer, say that you "
        "don't know. Use three sentences maximum and keep the answer concise."
        "\n\nContext:\n" + context + "\n\nQuestion:\n" + question
    )

    response = client.chat.completions.create(
        model="qwen/qwen3-32b",  # Updated Groq chat model to a supported one
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": question},
        ],
    )

    answer = response.choices[0].message.content
    return answer

# Example query
question = "tell me about databricks"
relevant_chunks = query_documents(question)
answer = generate_response(question, relevant_chunks)

print(answer)


==== Loading documents from directory ====
Loaded 9 documents


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<think>
Okay, the user asked to tell me about Databricks. Let me look through the provided context.

First, the context mentions that Databricks acquired Okera, a data governance platform focused on AI. So Databricks is involved in data governance and AI. They’re integrating Okera’s tech into their Unity Catalog, which is their existing governance solution for data and AI assets. Also, Databricks has its own LLM (large language model) and is enhancing governance with Okera's AI-powered systems for data classification and access controls. 

Another point is that Databricks highlighted shortcomings in traditional data governance for AI, especially with the rapid growth of assets and changing AI landscapes. They emphasized the need for governance that can handle dynamic environments. The acquisition of Okera was driven by Okera's isolation technology for governance without major overhead, even though it's in private preview. 

So, putting this together: Databricks is a company focused on 